Phase 1 - Data Ingestion

In [1]:
from langchain_community.document_loaders import (
    TextLoader,
    PyPDFLoader,
    CSVLoader,
    Docx2txtLoader
)

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

from dotenv import load_dotenv

import ipywidgets as widgets
from IPython.display import display

import os
import pickle
import hashlib


load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

if not GOOGLE_API_KEY:
    raise ValueError("GOOGLE_API_KEY not found in .env file")

C:\Users\pc\AppData\Local\Temp\ipykernel_3000\2812920043.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (


Document loader

In [2]:
def load_documents(file_path):

    file_type = os.path.splitext(file_path)[1].lower()

    if file_type == ".pdf":
        document = PyPDFLoader(file_path)

    elif file_type == ".txt":
        document = TextLoader(
            file_path,
            encoding="utf-8"
        )

    elif file_type == ".csv":
        document = CSVLoader(file_path)

    elif file_type == ".docx":
        document = Docx2txtLoader(file_path)

    else:
        raise ValueError(
            f"Unsupported file type: {file_type}"
        )

    return document.load()

Chunking

In [3]:
def chunk_documents(documents):

    document_split = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )

    chunks = document_split.split_documents(
        documents
    )

    os.makedirs(
        "../processed",
        exist_ok=True
    )

    with open(
        "../processed/chunks.pkl",
        "wb"
    ) as f:
        pickle.dump(chunks, f)

    return chunks

Embedding + Chroma

In [4]:
def create_vectorstore(chunks):

    embedding = GoogleGenerativeAIEmbeddings(
        model="gemini-embedding-001",
        google_api_key=GOOGLE_API_KEY
    )

    vectorstore = Chroma(
        collection_name="Neural",
        embedding_function=embedding,
        persist_directory="../chromadb"
    )

    ids = []

    for chunk in chunks:

        content = chunk.page_content
        source = chunk.metadata.get(
            "source",
            ""
        )

        document_id = hashlib.md5(
            f"{source}:{content}".encode("utf-8")
        ).hexdigest()

        ids.append(document_id)

    existing = vectorstore.get(
        ids=ids,
        include=[]
    )

    existing_ids = set(
        existing["ids"]
    )

    new_chunks = []
    new_ids = []

    for chunk, document_id in zip(
        chunks,
        ids
    ):

        if document_id not in existing_ids:
            new_chunks.append(chunk)
            new_ids.append(document_id)

    if new_chunks:

        vectorstore.add_documents(
            documents=new_chunks,
            ids=new_ids
        )

    return vectorstore

Upload

In [5]:
def ingest_file(file_path):

    documents = load_documents(
        file_path
    )

    chunks = chunk_documents(
        documents
    )

    vectorstore = create_vectorstore(
        chunks
    )

    return vectorstore

In [6]:
def upload_file():

    uploader = widgets.FileUpload(
        accept=".pdf,.txt,.csv,.docx",
        multiple=False
    )

    display(uploader)

    return uploader

In [7]:
def process_uploaded_file(uploader):

    if not uploader.value:
        raise ValueError(
            "No file uploaded."
        )

    uploaded_file = next(
        iter(uploader.value)
    )

    file_name = uploaded_file["name"]
    file_content = uploaded_file["content"]

    upload_folder = "../data/uploads"

    os.makedirs(
        upload_folder,
        exist_ok=True
    )

    file_path = os.path.join(
        upload_folder,
        file_name
    )

    with open(
        file_path,
        "wb"
    ) as f:
        f.write(file_content)

    vectorstore = ingest_file(
        file_path
    )

    return vectorstore

In [ ]:
uploader = upload_file()

In [ ]:
vectorstore = process_uploaded_file(
    uploader
)

print("File uploaded and indexed successfully!")